In [1]:
import pyodbc
import pandas as pd

In [2]:
def create_connection():
    # Thông tin kết nối SQL Server
    server = 'ZOHATEA'  # Tên server từ SSMS
    database = 'Customer'  # Đã sửa, bỏ dấu ;
    username = 'sa'
    password = 'ptit'

    # Kết nối SQL Server
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"  # Sử dụng driver có sẵn
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password}"
        )
        cursor = conn.cursor()
        print("Kết nối thành công!")
    except pyodbc.Error as ex:
        print(f"Lỗi kết nối: {ex}")
    return conn, cursor

# 2. DatPhong

In [3]:
df_datphong = pd.read_csv('../Customer/DatPhong.csv', encoding='utf-8', keep_default_na=False)
# chi lay hàng co ThoiGian >= '2025-06-19'
df_datphong = df_datphong[df_datphong['ThoiGian'] >= '2025-06-19']
df_datphong

,MaDatPhong,MaKH,ThoiGian,GhiChu
30165,DP30166,CUST01387,2025-06-19,Đặt giúp người khác
30166,DP30167,CUST00063,2025-06-19,Yêu cầu phòng yên tĩnh
30167,DP30168,CUST00230,2025-06-19,Yêu cầu phòng yên tĩnh
30168,DP30169,CUST02803,2025-06-19,
30169,DP30170,CUST00450,2025-06-19,
...,...,...,...,...
30887,DP30888,CUST00825,2025-07-01,Check-in sớm
30888,DP30889,CUST00747,2025-07-01,Yêu cầu phòng yên tĩnh
30889,DP30890,CUST01866,2025-07-01,Yêu cầu phòng yên tĩnh
30890,DP30891,CUST04852,2025-07-01,Check-in sớm


In [4]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DatPhong (MaDatPhong, MaKH, ThoiGian, GhiChu)
    VALUES (?, ?, ?, ?)
    ''', df_datphong.values.tolist())
    conn.commit()
    print(f"DatPhong: {len(df_datphong)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
DatPhong: 727 


# 3. PhongDuocDat

In [5]:
df_phongduocdat = pd.read_csv('../Customer/PhongDuocDat.csv', encoding='utf-8', keep_default_na=False)
df_phongduocdat = df_phongduocdat[df_phongduocdat['MaDatPhong'] >= df_datphong['MaDatPhong'].min()]
df_phongduocdat

,MaPhongDuocDat,MaDatPhong,MaPhong,NgayNhanPhong,NgayTraPhong,TongTienPhong
40646,PDD040647,DP30166,DNA002_R031,2025-06-19,2025-06-21,4000000
40647,PDD040648,DP30167,AGG005_R016,2025-06-19,2025-06-21,3400000
40648,PDD040649,DP30167,NTN002_R013,2025-06-19,2025-06-20,1800000
40649,PDD040650,DP30168,HPG001_R005,2025-06-19,2025-06-22,3600000
40650,PDD040651,DP30169,BDH001_R015,2025-06-19,2025-06-24,12000000
...,...,...,...,...,...,...
41621,PDD041622,DP30889,QNG001_R024,2025-07-01,2025-07-05,11200000
41622,PDD041623,DP30890,BTN001_R014,2025-07-01,2025-07-06,11000000
41623,PDD041624,DP30891,TBI002_R013,2025-07-01,2025-07-02,2400000
41624,PDD041625,DP30892,PYN002_R020,2025-07-01,2025-07-03,3000000


In [6]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO PhongDuocDat (MaPhongDuocDat, MaDatPhong, MaPhong, NgayNhanPhong, NgayTraPhong, TongTienPhong)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', df_phongduocdat.values.tolist())
    conn.commit()
    print(f"PhongDuocDat: {len(df_phongduocdat)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
PhongDuocDat: 980 


# 4. DichVu

In [7]:
df_dichvu = pd.read_csv('../Customer/DichVu.csv', encoding='utf-8', keep_default_na=False)
df_dichvu = df_dichvu[df_dichvu['ThoiGian'] >= '2025-06-19']
df_dichvu

,MaSuDungDichVu,MaDichVu,MaPhongDuocDat,ThoiGian,SoLuong,ThanhTien
101637,SDV101638,SVC010,PDD040705,2025-06-19,3,600000
101638,SDV101639,SVC023,PDD040663,2025-06-19,2,600000
101639,SDV101640,SVC018,PDD040693,2025-06-19,1,2000000
101640,SDV101641,SVC006,PDD040684,2025-06-19,2,600000
101641,SDV101642,SVC022,PDD040632,2025-06-19,3,0
...,...,...,...,...,...,...
104319,SDV104320,SVC017,PDD041605,2025-07-04,1,500000
104320,SDV104321,SVC009,PDD041536,2025-07-04,2,0
104321,SDV104322,SVC002,PDD041603,2025-07-04,3,360000
104322,SDV104323,SVC023,PDD041623,2025-07-05,2,600000


In [8]:
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DichVu (MaDichVuSuDung, MaDichVu, MaPhongDuocDat, ThoiGian, SoLuong, ThanhTien)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', df_dichvu.values.tolist())
    conn.commit()
    print(f"DichVu: {len(df_dichvu)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
DichVu: 2687 


# 5. DanhGia

In [9]:
df_danhgia = pd.read_csv('../Customer/DanhGia.csv', encoding='utf-8', keep_default_na=False)
df_danhgia = df_danhgia[df_danhgia['ThoiGian'] >='2025-06-19']
df_danhgia

,MaDanhGia,MaPhongDuocDat,ThoiGian,DiemDanhGia,NhanXet
36507,DGPDD040407_1,PDD040407,2025-06-19,2,Không hài lòng lắm.
36508,DGPDD040481_1,PDD040481,2025-06-19,5,Dịch vụ hoàn hảo.
36509,DGPDD040475_1,PDD040475,2025-06-19,5,Tuyệt vời!
36510,DGPDD040333_1,PDD040333,2025-06-19,4,"Nhân viên thân thiện, phòng sạch sẽ."
36511,DGPDD040396_1,PDD040396,2025-06-19,4,"Tốt, nhưng còn vài điểm cần cải thiện."
...,...,...,...,...,...
37709,DGPDD041605_1,PDD041605,2025-07-08,3,"Phục vụ trung bình, giá hợp lý."
37710,DGPDD041626_2,PDD041626,2025-07-08,4,"Nhân viên thân thiện, phòng sạch sẽ."
37711,DGPDD041591_1,PDD041591,2025-07-09,5,Dịch vụ hoàn hảo.
37712,DGPDD041619_1,PDD041619,2025-07-09,3,"Tạm ổn, cần nâng cấp một số tiện nghi."


In [10]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DanhGia (MaDanhGia, MaPhongDuocDat, ThoiGian, DiemDanhGia, NhanXet)
    VALUES (?, ?, ?, ?, ?)
    ''', df_danhgia.values.tolist())
    conn.commit()
    print(f"DanhGia: {len(df_danhgia)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
DanhGia: 1207 
